In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import os
import json
import time
import argparse
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from PIL import Image

"""
Stage 1 — Model Implementation
================================
Model    : EfficientNet-B0  (pretrained on ImageNet, fine-tuned)
Task     : Multi-class leaf disease classification
Classes  : 13 disease categories
Dataset  : dataset_ready/  (from preprocessing pipeline)

Output:
  models/classifier.pth        ← best model checkpoint
  models/class_map.json         ← class index mapping
  models/training_log.json      ← epoch-wise metrics
  models/confusion_matrix.png   ← test set confusion matrix
  models/classification_report.txt ← precision, recall, F1 per class

Run:
  python stage1_model.py --mode train       # train the model
  python stage1_model.py --mode evaluate    # evaluate saved model on test set
  python stage1_model.py --mode predict --image path/to/leaf.jpg  # single image

Dependencies:
  pip install torch torchvision matplotlib seaborn scikit-learn
"""

import os
import json
import time
import argparse
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from PIL import Image

# ══════════════════════════════════════════════════════════════════════════════
# ❶  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

DATASET_DIR  = "dataset_ready"
MODEL_DIR    = "models"
MODEL_PATH   = f"{MODEL_DIR}/classifier.pth"
CLASS_MAP    = f"{MODEL_DIR}/class_map.json"
LOG_PATH     = f"{MODEL_DIR}/training_log.json"
CM_PATH      = f"{MODEL_DIR}/confusion_matrix.png"
REPORT_PATH  = f"{MODEL_DIR}/classification_report.txt"

IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-4
NUM_WORKERS  = 4
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

CLASSES = [
    "Anthracnose", "BacterialBlight", "BlackSpot", "Canker",
    "CurlVirus", "DeficiencyLeaf", "DryLeaf", "Greening",
    "Healthy", "LeafMiner", "Melanose", "SootyMould", "SpiderMites"
]

# Try dataset-specific normalisation, fall back to ImageNet defaults
try:
    from normalization_stats import MEAN, STD
    print("✓ Using dataset-specific normalisation stats")
except ImportError:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print("⚠ Using ImageNet normalisation defaults")

# ══════════════════════════════════════════════════════════════════════════════
# ❷  TRANSFORMS
# ══════════════════════════════════════════════════════════════════════════════

def get_transforms(split: str) -> transforms.Compose:
    normalize = transforms.Normalize(mean=MEAN, std=STD)

    if split == "train":
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(
                brightness=0.25, contrast=0.25,
                saturation=0.20, hue=0.08
            ),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            normalize,
        ])

# ══════════════════════════════════════════════════════════════════════════════
# ❸  DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

def make_weighted_sampler(dataset) -> WeightedRandomSampler:
    """
    Each class is sampled equally regardless of count.
    Handles residual imbalance after balancing step.
    """
    class_counts = torch.zeros(len(dataset.classes))
    for _, label in dataset.samples:
        class_counts[label] += 1

    class_weights  = 1.0 / class_counts
    sample_weights = [class_weights[label] for _, label in dataset.samples]

    return WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(sample_weights),
        replacement = True,
    )


def get_dataloaders():
    loaders = {}
    for split in ["train", "val", "test"]:
        ds = datasets.ImageFolder(
            root      = str(Path(DATASET_DIR) / split),
            transform = get_transforms(split),
        )
        if split == "train":
            sampler = make_weighted_sampler(ds)
            loader  = DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                                 num_workers=NUM_WORKERS, pin_memory=True)
        else:
            loader  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)
        loaders[split] = (ds, loader)
        print(f"  {split:5s}: {len(ds):5d} images  {len(loader):4d} batches")

    return loaders

# ══════════════════════════════════════════════════════════════════════════════
# ❹  MODEL
# ══════════════════════════════════════════════════════════════════════════════

def build_model(num_classes: int) -> nn.Module:
    """
    EfficientNet-B0 with ImageNet weights.
    Replace final classifier head with:
      Dropout(0.3) → Linear(1280 → num_classes)
    """
    # Bypass SSL verification for weight download
    import urllib.request
    import ssl
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
    urllib.request.install_opener(opener)
    
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
    )
    # Freeze backbone initially
    for p in model.features.parameters():
        p.requires_grad = False

    in_feats = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_feats, num_classes),
    )
    return model.to(DEVICE)

# ══════════════════════════════════════════════════════════════════════════════
# ❺  TRAIN / EVAL LOOPS
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = correct = total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

# ══════════════════════════════════════════════════════════════════════════════
# ❻  TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def train():
    os.makedirs(MODEL_DIR, exist_ok=True)
    print(f"\nDevice : {DEVICE}")
    print(f"Dataset: {DATASET_DIR}\n")

    print("Loading datasets...")
    loaders     = get_dataloaders()
    train_ds, train_loader = loaders["train"]
    _,         val_loader  = loaders["val"]
    num_classes = len(train_ds.classes)

    # Save class map
    class_to_idx = train_ds.class_to_idx
    idx_to_class = {str(v): k for k, v in class_to_idx.items()}
    with open(CLASS_MAP, "w") as f:
        json.dump({"class_to_idx": class_to_idx,
                   "idx_to_class": idx_to_class}, f, indent=2)
    print(f"\n  Classes ({num_classes}): {list(class_to_idx.keys())}")

    model     = build_model(num_classes)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ── Phase 1: head only (5 epochs) ────────────────────────────────────────
    head_opt = torch.optim.Adam(model.classifier.parameters(), lr=LR)
    print(f"\n{'─'*75}")
    print(f"  Phase 1 — head only (5 epochs, backbone frozen)")
    print(f"{'─'*75}")
    print(f"  {'Ep':>3}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")

    for epoch in range(1, 6):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, head_opt)
        va_loss, va_acc = evaluate_loader(model, val_loader,   criterion)
        print(f"  {epoch:>3}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  "
              f"{va_loss:>8.4f}  {va_acc:>7.4f}  ({time.time()-t0:.1f}s)")

    # ── Phase 2: full fine-tune ───────────────────────────────────────────────
    for p in model.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR / 5, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=EPOCHS - 5, eta_min=1e-6)

    best_val_acc = 0.0
    log = []
    remaining = EPOCHS - 5

    print(f"\n{'─'*75}")
    print(f"  Phase 2 — full fine-tune ({remaining} epochs, all layers)")
    print(f"{'─'*75}")
    print(f"  {'Ep':>3}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")

    for epoch in range(1, remaining + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate_loader(model, val_loader,   criterion)
        scheduler.step()

        marker = ""
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save({
                "epoch":       epoch + 5,
                "model_state": model.state_dict(),
                "num_classes": num_classes,
                "class_to_idx": class_to_idx,
                "best_val_acc": best_val_acc,
            }, MODEL_PATH)
            marker = "  ← saved"

        print(f"  {epoch:>3}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  "
              f"{va_loss:>8.4f}  {va_acc:>7.4f}  "
              f"({time.time()-t0:.1f}s){marker}")

        log.append({"epoch": epoch + 5,
                    "train_loss": tr_loss, "train_acc": tr_acc,
                    "val_loss":   va_loss, "val_acc":   va_acc})

    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)

    print(f"\n  Best val accuracy : {best_val_acc:.4f}")
    print(f"  Model saved       : {MODEL_PATH}")
    print(f"  Training log      : {LOG_PATH}")
    print("\nNow run: python stage1_model.py --mode evaluate")

# ══════════════════════════════════════════════════════════════════════════════
# ❼  EVALUATION  (confusion matrix + classification report)
# ══════════════════════════════════════════════════════════════════════════════

def evaluate():
    print(f"\nLoading model from {MODEL_PATH} ...")
    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
    model = build_model(ckpt["num_classes"])
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    with open(CLASS_MAP) as f:
        cmap = json.load(f)
    idx_to_class = {int(k): v for k, v in cmap["idx_to_class"].items()}
    class_names  = [idx_to_class[i] for i in range(len(idx_to_class))]

    # Load test set
    test_ds = datasets.ImageFolder(
        root      = str(Path(DATASET_DIR) / "test"),
        transform = get_transforms("test"),
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS)

    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(DEVICE)
            preds  = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ── Overall accuracy ──────────────────────────────────────────────────────
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n  Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Tested on     : {len(all_labels)} images")
    print(f"  Best val acc  : {ckpt['best_val_acc']:.4f} (epoch {ckpt['epoch']})")

    # ── Classification report ─────────────────────────────────────────────────
    report = classification_report(all_labels, all_preds,
                                   target_names=class_names, digits=4)
    print(f"\n{report}")
    with open(REPORT_PATH, "w") as f:
        f.write(f"Test Accuracy: {acc:.4f}\n\n")
        f.write(report)
    print(f"  ✓ Report saved → {REPORT_PATH}")

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds)
    plot_confusion_matrix(cm, class_names)

    return acc, report


def plot_confusion_matrix(cm: np.ndarray, class_names: list):
    fig, ax = plt.subplots(figsize=(14, 12))

    # Normalise row-wise so colour shows recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm,
        annot      = cm,           # show raw counts
        fmt        = "d",
        cmap       = "Blues",
        xticklabels= class_names,
        yticklabels= class_names,
        linewidths = 0.5,
        ax         = ax,
        cbar_kws   = {"label": "Recall (row-normalised)"},
    )
    ax.set_xlabel("Predicted label", fontsize=13)
    ax.set_ylabel("True label",      fontsize=13)
    ax.set_title("Confusion Matrix — Stage 1 Test Set", fontsize=15, pad=15)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(rotation=0,  fontsize=10)
    plt.tight_layout()
    plt.savefig(CM_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Confusion matrix saved → {CM_PATH}")


def plot_training_curves():
    """Plot train/val loss and accuracy curves from training log."""
    if not Path(LOG_PATH).exists():
        print(f"No training log found at {LOG_PATH}")
        return

    with open(LOG_PATH) as f:
        log = json.load(f)

    epochs    = [e["epoch"]      for e in log]
    tr_loss   = [e["train_loss"] for e in log]
    va_loss   = [e["val_loss"]   for e in log]
    tr_acc    = [e["train_acc"]  for e in log]
    va_acc    = [e["val_acc"]    for e in log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, tr_loss, label="Train loss", marker="o", markersize=4)
    ax1.plot(epochs, va_loss, label="Val loss",   marker="o", markersize=4)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Loss curves"); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, tr_acc, label="Train acc", marker="o", markersize=4)
    ax2.plot(epochs, va_acc, label="Val acc",   marker="o", markersize=4)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy curves"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle("Stage 1 — EfficientNet-B0 Training", fontsize=14)
    plt.tight_layout()
    curve_path = f"{MODEL_DIR}/training_curves.png"
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Training curves saved → {curve_path}")

# ══════════════════════════════════════════════════════════════════════════════
# ❽  SINGLE IMAGE PREDICTION
# ══════════════════════════════════════════════════════════════════════════════

def predict(image_path: str):
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    with open(CLASS_MAP) as f:
        cmap = json.load(f)
    idx_to_class = {int(k): v for k, v in cmap["idx_to_class"].items()}

    model = build_model(ckpt["num_classes"])
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    tf  = get_transforms("test")
    img = Image.open(image_path).convert("RGB")
    inp = tf(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(inp)
        probs  = torch.softmax(logits, dim=1)[0]
        conf, idx = probs.max(0)

    disease    = idx_to_class[idx.item()]
    confidence = conf.item() * 100

    # Top 3 predictions
    top3 = probs.topk(3)
    print(f"\n  Image     : {image_path}")
    print(f"  Prediction: {disease}")
    print(f"  Confidence: {confidence:.1f}%")
    print(f"\n  Top 3:")
    for score, cls_idx in zip(top3.values, top3.indices):
        print(f"    {idx_to_class[cls_idx.item()]:20s}: {score.item()*100:.1f}%")

    return {"disease": disease, "confidence": f"{confidence:.1f}%"}

# ══════════════════════════════════════════════════════════════════════════════
# ❾  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import sys
    sys.argv = [sys.argv[0]]   # fix Jupyter argparse conflict

    parser = argparse.ArgumentParser()
    parser.add_argument("--mode",  choices=["train", "evaluate", "predict"],
                        default="train")
    parser.add_argument("--image", help="Image path for predict mode")
    args = parser.parse_args()

    if args.mode == "train":
        train()
        print("\nRunning evaluation on test set...")
        evaluate()
        plot_training_curves()

    elif args.mode == "evaluate":
        evaluate()
        plot_training_curves()

    elif args.mode == "predict":
        if not args.image:
            print("Provide --image path/to/leaf.jpg")
        else:
            predict(args.image)

9.2%

✓ Using dataset-specific normalisation stats

Device : cpu
Dataset: dataset_ready

Loading datasets...
  train:  7069 images   221 batches
  val  :  1076 images    34 batches
  test :  1088 images    34 batches

  Classes (13): ['Anthracnose', 'BacterialBlight', 'BlackSpot', 'Canker', 'CurlVirus', 'DeficiencyLeaf', 'DryLeaf', 'Greening', 'Healthy', 'LeafMiner', 'Melanose', 'SootyMould', 'SpiderMites']
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/udayapragnagangavaram/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100.0%
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



───────────────────────────────────────────────────────────────────────────
  Phase 1 — head only (5 epochs, backbone frozen)
───────────────────────────────────────────────────────────────────────────
   Ep   Tr Loss   Tr Acc   Va Loss   Va Acc
    1    2.2864   0.3515    1.9847   0.5325  (385.1s)
    2    1.8880   0.5719    1.7088   0.6059  (384.2s)
    3    1.6643   0.6360    1.5645   0.6227  (381.6s)
    4    1.5394   0.6540    1.4777   0.6431  (378.7s)
    5    1.4738   0.6671    1.4280   0.6608  (2698.8s)

───────────────────────────────────────────────────────────────────────────
  Phase 2 — full fine-tune (15 epochs, all layers)
───────────────────────────────────────────────────────────────────────────
   Ep   Tr Loss   Tr Acc   Va Loss   Va Acc
    1    1.2422   0.7455    1.2548   0.7045  (1377.9s)  ← saved
    2    1.0993   0.7779    1.1725   0.7361  (1384.8s)  ← saved
    3    1.0062   0.8244    1.1132   0.7500  (1376.1s)  ← saved
    4    0.9620   0.8389    1.0865   0.766

In [2]:
!pip install torch torchvision matplotlib seaborn scikit-learn